# U-Net Baseline — Google Colab (A100, 512×512)

**Drive layout** (same as `WWR_Seg_Model.ipynb`):

| Path | Content |
|------|----------|
| `My Drive/WWR_Seg_Model/data.zip` | Training data |
| `/content/data/` | Local cache (fast I/O) |
| `My Drive/WWR_Seg_Model/models/` | Saved models |
| `My Drive/WWR_Seg_Model/results/unet/` | Plots & outputs |

1. **Runtime → A100 GPU**
2. Run all cells in order


In [ ]:
!pip install -q tensorflow>=2.15 opencv-python-headless scikit-learn tqdm matplotlib


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

from google.colab import drive

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/WWR_Seg_Model')
DATA_ZIP = DRIVE_BASE / 'data.zip'
LOCAL_DATA = Path('/content/data')
MODELS_DIR = DRIVE_BASE / 'models'
RESULTS_DIR = DRIVE_BASE / 'results' / 'unet'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Drive base :', DRIVE_BASE)
print('Data zip   :', DATA_ZIP, '→', DATA_ZIP.exists())


In [ ]:
IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}


def _count_images(folder: Path) -> int:
    return sum(1 for f in folder.iterdir() if f.suffix.lower() in IMAGE_EXTS)


def prepare_local_data(force: bool = False) -> tuple[Path, Path]:
    """Unzip data.zip to /content/data and normalize to train/images + train/masks."""
    train_img = LOCAL_DATA / 'train' / 'images'
    train_msk = LOCAL_DATA / 'train' / 'masks'
    if not force and train_img.exists() and _count_images(train_img) > 0:
        print('Using cached local data:', train_img)
        return train_img, train_msk

    if not DATA_ZIP.exists():
        raise FileNotFoundError(f'Upload data.zip to {DATA_ZIP}')

    tmp = LOCAL_DATA / '_zip_extract'
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)

    print('Extracting', DATA_ZIP)
    with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
        zf.extractall(tmp)

    flat_img = tmp / 'images'
    flat_msk = tmp / 'masks'
    if flat_img.exists() and flat_msk.exists():
        train_img.parent.mkdir(parents=True, exist_ok=True)
        if LOCAL_DATA.exists():
            shutil.rmtree(LOCAL_DATA / 'train', ignore_errors=True)
        shutil.copytree(flat_img, train_img)
        shutil.copytree(flat_msk, train_msk)
    elif (tmp / 'train' / 'images').exists():
        if LOCAL_DATA.exists():
            shutil.rmtree(LOCAL_DATA, ignore_errors=True)
        shutil.copytree(tmp, LOCAL_DATA)
    else:
        raise FileNotFoundError('Expected images/ + masks/ or train/images inside data.zip')

    shutil.rmtree(tmp, ignore_errors=True)
    print('Train images:', _count_images(train_img))
    return train_img, train_msk


images_dir, masks_dir = prepare_local_data(force=False)
print('Images:', images_dir)
print('Masks :', masks_dir)


In [ ]:
import glob
import cv2
import numpy as np
import tqdm
import tensorflow as tf
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras import Model, Input
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import (
    BatchNormalization, Concatenate, Conv2D, Conv2DTranspose, Dropout,
)
from tensorflow.keras.metrics import MeanIoU

OUTPUT_CHANNELS = 4
SIZE_X = 512
SIZE_Y = 512
MASK_LUT = {72: 0, 128: 1, 220: 2, 255: 3}  # Roof, Window, Wall, Other


def pair_image_mask_paths(img_dir: Path, mask_dir: Path) -> list[tuple[str, str]]:
    pairs = []
    for img_path in sorted(img_dir.iterdir()):
        if img_path.suffix.lower() not in {'.png', '.jpg', '.jpeg'}:
            continue
        stem = img_path.stem
        mask_stem = stem.replace('_texture', '_mask') if stem.endswith('_texture') else f'{stem}_mask'
        mask_path = mask_dir / f'{mask_stem}{img_path.suffix}'
        if not mask_path.exists():
            raise FileNotFoundError(f'No mask for {img_path.name}')
        pairs.append((str(img_path), str(mask_path)))
    return pairs


def standardize(x):
    x = np.array(x, dtype='float32')
    x -= np.min(x)
    denom = np.percentile(x, 98)
    x /= denom if denom > 0 else 1.0
    x[x > 1] = 1
    return x


def preprocessing(img):
    image = np.array(img)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    image = np.stack([gray, gray, gray], axis=-1)
    return standardize(image)


pairs = pair_image_mask_paths(images_dir, masks_dir)
print(f'Paired samples: {len(pairs)}')
print('First pair:', pairs[0])


In [ ]:
train_images, train_masks = [], []

for img_path, mask_path in tqdm.tqdm(pairs, desc='Loading'):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (SIZE_X, SIZE_Y))
    train_images.append(preprocessing(img))

    mask0 = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask1 = cv2.resize(mask0, (SIZE_X, SIZE_Y), interpolation=cv2.INTER_NEAREST)
    for old_val, new_val in MASK_LUT.items():
        mask1[mask1 == old_val] = new_val
    train_masks.append(mask1)

train_images = np.array(train_images, dtype=np.float32)
train_masks = np.array(train_masks, dtype=np.int32)

X_train, X_val, y_train, y_val = train_test_split(
    train_images, train_masks, test_size=0.10, random_state=42, shuffle=True
)
print('Train:', len(X_train), '| Val:', len(X_val))
print('Classes:', np.unique(y_train))
# Keep integer masks (H, W) — required for correct MeanIoU


In [ ]:
def augment(image, mask):
    mask = tf.cast(mask, tf.int32)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        mask = tf.image.flip_left_right(tf.expand_dims(mask, -1))[..., 0]
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, mask


def build_dataset(X, Y, training=True, batch_size=8):
    ds = tf.data.Dataset.from_tensor_slices((X, Y))
    if training:
        ds = ds.shuffle(len(X)).map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


In [ ]:
def upsample_with_dropout(filters, size, dropout_rate=0.2):
    return tf.keras.Sequential([
        Conv2DTranspose(filters, size, strides=2, padding='same', use_bias=False),
        BatchNormalization(),
        Dropout(dropout_rate),
        tf.keras.layers.ReLU(),
    ])


def dice_loss(y_true, y_pred, smooth=1e-6):
  # y_true: (B, H, W) int  |  y_pred: (B, H, W, C) softmax
    y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), OUTPUT_CHANNELS)
    y_true_f = tf.reshape(y_true_oh, [-1, OUTPUT_CHANNELS])
    y_pred_f = tf.reshape(y_pred, [-1, OUTPUT_CHANNELS])
    intersection = tf.reduce_sum(y_true_f * y_pred_f, axis=0)
    denominator = tf.reduce_sum(y_true_f, axis=0) + tf.reduce_sum(y_pred_f, axis=0)
    dice = (2.0 * intersection + smooth) / (denominator + smooth)
    return 1.0 - tf.reduce_mean(dice)


def combined_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    cce = tf.keras.losses.SparseCategoricalCrossentropy()(y_true, y_pred)
    return cce + dice_loss(y_true, y_pred)


def unet_model(output_channels=4, dropout_rate=0.2, fine_tune_at=100):
    h, w, c = X_train.shape[1:]
    base = MobileNetV2(input_shape=[h, w, c], include_top=False, weights='imagenet')
    layer_names = [
        'block_1_expand_relu', 'block_3_expand_relu', 'block_6_expand_relu',
        'block_13_expand_relu', 'block_16_project',
    ]
    down_stack = Model(base.input, [base.get_layer(n).output for n in layer_names])
    for layer in down_stack.layers:
        layer.trainable = False
    for layer in down_stack.layers[fine_tune_at:]:
        layer.trainable = True

    up_stack = [
        upsample_with_dropout(512, 3, dropout_rate),
        upsample_with_dropout(256, 3, dropout_rate),
        upsample_with_dropout(128, 3, dropout_rate),
        upsample_with_dropout(64, 3, dropout_rate),
    ]
    inputs = Input(shape=[h, w, c])
    skips = down_stack(inputs)
    x = skips[-1]
    for up, skip in zip(up_stack, reversed(skips[:-1])):
        x = up(x)
        x = Concatenate()([x, skip])
    outputs = Conv2DTranspose(output_channels, 3, strides=2, padding='same', activation='softmax')(x)
    return Model(inputs, outputs)


In [ ]:
def create_mask(pred_mask):
    return tf.argmax(pred_mask, axis=-1)[0].numpy()


def show_predictions(epoch, sample_idx=50):
    fig = plt.figure(figsize=(12, 4))
    for col, (title, arr) in enumerate([
        ('Input', X_val[sample_idx]),
        ('Ground Truth', y_val[sample_idx]),
        ('Prediction', create_mask(model.predict(X_val[sample_idx:sample_idx+1], verbose=0))),
    ]):
        ax = fig.add_subplot(1, 3, col + 1)
        ax.imshow(arr if col == 0 else arr, cmap=None if col == 0 else 'jet')
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f'Epoch {epoch + 1}')
    out = RESULTS_DIR / f'epoch_{epoch + 1:03d}.png'
    fig.savefig(out, dpi=100, bbox_inches='tight')
    plt.show()


class DisplayCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        show_predictions(epoch)


In [ ]:
model = unet_model(OUTPUT_CHANNELS, dropout_rate=0.2, fine_tune_at=100)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss,
    metrics=['accuracy', MeanIoU(num_classes=OUTPUT_CHANNELS, name='mean_iou')],
)
model.summary()


In [ ]:
EPOCHS = 30
BATCH_SIZE = 8  # 512×512 on A100

train_ds = build_dataset(X_train, y_train, training=True, batch_size=BATCH_SIZE)
val_ds = build_dataset(X_val, y_val, training=False, batch_size=BATCH_SIZE)

checkpoint_path = MODELS_DIR / 'unet_baseline.keras'
callbacks = [
    DisplayCallback(),
    tf.keras.callbacks.ModelCheckpoint(
        str(checkpoint_path), monitor='val_mean_iou', mode='max',
        save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(monitor='val_mean_iou', mode='max', patience=8, restore_best_weights=True),
]

history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
print('Best model →', checkpoint_path)


In [ ]:
iou_key = 'mean_iou' if 'mean_iou' in history.history else 'mean_io_u'
val_iou_key = 'val_' + iou_key
plt.plot(history.history[iou_key], label='train IoU')
plt.plot(history.history[val_iou_key], label='val IoU')
plt.xlabel('Epoch')
plt.legend()
plt.savefig(RESULTS_DIR / 'training_curve.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# Optional test images (upload to Drive when ready)
TEST_IMG_DIR = DRIVE_BASE / 'test_images' / 'final_images'

if TEST_IMG_DIR.exists():
    test_paths = sorted(TEST_IMG_DIR.glob('*.png'))
    print(f'Test images: {len(test_paths)}')
    img = cv2.imread(str(test_paths[0]))
    img = cv2.resize(img, (SIZE_X, SIZE_Y))
    img = preprocessing(img)
    pred = create_mask(model.predict(img[np.newaxis, ...], verbose=0))
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1); plt.imshow(img); plt.title('Input'); plt.axis('off')
    plt.subplot(1, 2, 2); plt.imshow(pred, cmap='jet'); plt.title('Prediction'); plt.axis('off')
    plt.savefig(RESULTS_DIR / 'test_sample.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No test set yet →', TEST_IMG_DIR)
